In [1]:
!nvidia-smi

Wed Oct  7 06:21:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**CELL 1: downlaod and explore folders**

In [2]:
import kagglehub, os, glob
path = kagglehub.competition_download('shl-hiring-assessment-2026')
print("PATH:", path)

for root, dirs, files in os.walk(path):
    depth = root.replace(path, "").count(os.sep)
    if depth > 2: continue
    print("  " * depth + os.path.basename(root) + "/", f"({len(files)} files)")
    for f in files[:5]:
        print("  " * (depth + 1) + f)

PATH: /kaggle/input/competitions/shl-hiring-assessment-2026
shl-hiring-assessment-2026/ (0 files)
  Dataset_Final/ (3 files)
    sample_submission.csv
    train.csv
    test.csv
    test/ (216 files)
      audio_49.wav
      audio_90.wav
      audio_77.wav
      audio_66.wav
      audio_54.wav
    train/ (769 files)
      audio_49.wav
      audio_663.wav
      audio_5059.wav
      audio_90.wav
      audio_581.wav


**CELL 2: read the CSVs**

In [3]:
import pandas as pd
for f in glob.glob(f"{path}/**/*.csv", recursive=True):
    df = pd.read_csv(f)
    print("\n==", f, df.shape)
    print(df.head())
    print(df.dtypes)


== /kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/sample_submission.csv (204, 2)
         filename  label
0   audio_804.wav     -1
1  audio_1028.wav     -1
2   audio_865.wav     -1
3   audio_774.wav     -1
4  audio_1138.wav     -1
filename    object
label        int64
dtype: object

== /kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/train.csv (769, 2)
        filename  label
0  audio_192.wav    3.0
1  audio_436.wav    3.0
2  audio_447.wav    3.5
3  audio_126.wav    2.0
4   audio_61.wav    2.0
filename     object
label       float64
dtype: object

== /kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test.csv (216, 2)
        filename  label
0  audio_128.wav     -1
1  audio_156.wav     -1
2   audio_19.wav     -1
3  audio_108.wav     -1
4  audio_206.wav     -1
filename    object
label        int64
dtype: object


**CELL 3: Data audit**

In [4]:
import soundfile as sf, numpy as np
# print("path:", path)
train = pd.read_csv(f"{path}/Dataset_Final/train.csv")   # fix name
print(train.describe())
print(train.iloc[:, -1].value_counts().sort_index())  # label dist

audio_files = glob.glob(f"{path}/**/*.wav", recursive=True) + glob.glob(f"{path}/**/*.mp3", recursive=True)
print(len(audio_files), "audio files")
info = [sf.info(a) for a in audio_files[:50]]
print({i.samplerate for i in info}, {i.channels for i in info})
print("dur min/mean/max:", min(i.duration for i in info), np.mean([i.duration for i in info]), max(i.duration for i in info))

            label
count  769.000000
mean     3.313394
std      1.239000
min      0.000000
25%      2.500000
50%      3.000000
75%      4.000000
max      5.000000
label
0.0     37
1.0      1
1.5      3
2.0    102
2.5     79
3.0    174
3.5     64
4.0    124
4.5     52
5.0    133
Name: count, dtype: int64
985 audio files
{16000} {1}
dur min/mean/max: 6.48 49.99579125 60.08


**CELL A: sanity checks**

In [5]:
D = f"{path}/Dataset_Final"
train = pd.read_csv(f"{D}/train.csv")
test  = pd.read_csv(f"{D}/test.csv")
sub   = pd.read_csv(f"{D}/sample_submission.csv")

# submission vs test mismatch
print("sub in test:", sub.filename.isin(test.filename).sum(), "/", len(sub))
print("test not in sub:", (~test.filename.isin(sub.filename)).sum())
print("dups test:", test.filename.duplicated().sum(), "dups sub:", sub.filename.duplicated().sum())
print("train files exist:", all(os.path.exists(f"{D}/train/{f}") for f in train.filename))
print("test files exist:", all(os.path.exists(f"{D}/test/{f}") for f in test.filename))

# durations all files + label-0 inspection
train["dur"] = [sf.info(f"{D}/train/{f}").duration for f in train.filename]
train["rms"] = [np.sqrt(np.mean(sf.read(f"{D}/train/{f}")[0]**2)) for f in train.filename]
print(train.groupby("label")[["dur","rms"]].mean())
print(train[train.label==0][["filename","dur","rms"]].head(10))

sub in test: 25 / 204
test not in sub: 191
dups test: 0 dups sub: 0
train files exist: True
test files exist: True
             dur       rms
label                     
0.0    57.864160  0.121955
1.0    45.060000  0.031038
1.5    60.069792  0.046961
2.0    53.561077  0.050991
2.5    55.125922  0.055948
3.0    55.726029  0.059574
3.5    56.648949  0.057599
4.0    52.809619  0.057546
4.5    58.391822  0.057756
5.0    58.681128  0.061202
           filename        dur       rms
17   audio_5055.wav  60.080000  0.081792
30   audio_5066.wav  60.074688  0.160260
41   audio_5069.wav  60.074688  0.114312
49   audio_5073.wav  60.074688  0.128264
51   audio_5039.wav  60.074688  0.066970
73   audio_5056.wav  60.080000  0.096729
96   audio_5062.wav  61.034687  0.153744
111  audio_5037.wav  45.060000  0.126428
122  audio_5054.wav  58.666688  0.129157
133  audio_5057.wav  60.074688  0.127552


In [6]:
!pip -q install faster-whisper


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 75.2 MB/s eta 0:00:00


**CELL B: transcription (Whisper medium via faster-whisper)**

In [7]:
import json, torch, soundfile as sf, numpy as np
from faster_whisper import WhisperModel
from tqdm.auto import tqdm

OUT = "/kaggle/working/transcripts"; os.makedirs(OUT, exist_ok=True)
model = WhisperModel("medium", device="cuda", compute_type="float16")

PROMPT = "Um, so, uh, I think that, you know, like, it was, was good."

def transcribe(split, files):
    for f in tqdm(files):
        o = f"{OUT}/{split}_{f}.json"
        if os.path.exists(o): continue
        audio, sr = sf.read(f"{D}/{split}/{f}", dtype="float32")
        if audio.ndim > 1: audio = audio.mean(1)
        assert sr == 16000, (f, sr)
        segs, info = model.transcribe(
            audio, beam_size=5, word_timestamps=True,
            vad_filter=False, condition_on_previous_text=False,
            initial_prompt=PROMPT,
        )
        words, text = [], []
        for s in segs:
            text.append(s.text)
            for w in (s.words or []):
                words.append([w.word, w.start, w.end, w.probability])
        json.dump({"lang": info.language, "lang_p": info.language_probability,
                   "text": " ".join(text).strip(), "words": words}, open(o, "w"))

transcribe("train", train.filename)
transcribe("test",  test.filename)

  0%|          | 0/769 [00:00<?, ?it/s]

  0%|          | 0/216 [00:00<?, ?it/s]

**CELL C: WavLM audio Embedding**

In [8]:
import torch, numpy as np, soundfile as sf
from transformers import AutoFeatureExtractor, WavLMModel

dev = "cuda:1" if torch.cuda.device_count() > 1 else "cuda"
fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
wm = WavLMModel.from_pretrained("microsoft/wavlm-base-plus").to(dev).half().eval()

def emb(p):
    a, sr = sf.read(p, dtype="float32")
    if a.ndim > 1: a = a.mean(1)
    x = fe(a, sampling_rate=16000, return_tensors="pt").input_values.to(dev).half()
    with torch.no_grad():
        hs = wm(x, output_hidden_states=True).hidden_states
    h = torch.stack(hs).squeeze(1).float()               # 13,T,768
    return torch.cat([h.mean(1), h.std(1)], -1).cpu().numpy()  # 13,1536

for split, df in [("train", train), ("test", test)]:
    E = np.stack([emb(f"{D}/{split}/{f}") for f in tqdm(df.filename)])
    np.save(f"/kaggle/working/wavlm_{split}.npy", E); print(split, E.shape)

preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.23k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  378MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

...

model.safetensors: downloading bytes:           |  0.00B            

  0%|          | 0/769 [00:00<?, ?it/s]

train (769, 13, 1536)


  0%|          | 0/216 [00:00<?, ?it/s]

test (216, 13, 1536)


**CELL D: handcrafted features**
loading GPT 2 model here 


In [9]:
import re, json
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

tok = GPT2TokenizerFast.from_pretrained("gpt2")
lm = GPT2LMHeadModel.from_pretrained("gpt2").to(dev).eval()
FILL = {"um","uh","er","erm","ah","hmm","mm","uhm"}

def ppl(text):
    ids = tok(text, return_tensors="pt", truncation=True, max_length=1024).input_ids.to(dev)
    if ids.shape[1] < 2: return np.nan, np.nan
    with torch.no_grad(): out = lm(ids, labels=ids)
    return float(out.loss), float(torch.exp(out.loss))

def feats(split, f):
    j = json.load(open(f"{OUT}/{split}_{f}.json"))
    W = j["words"]; text = j["text"]
    toks = [re.sub(r"[^a-z']", "", w[0].lower()) for w in W]
    toks = [t for t in toks if t]
    n = len(toks); dur = (W[-1][2] - W[0][1]) if n > 1 else 1e-3
    gaps = np.array([W[i+1][1] - W[i][2] for i in range(len(W)-1)]) if n > 1 else np.array([0.])
    pr = np.array([w[3] for w in W]) if W else np.array([0.])
    sents = [s for s in re.split(r"[.!?]+", text) if s.strip()]
    nll, pp = ppl(text) if n > 1 else (np.nan, np.nan)
    return dict(
        n_words=n, wpm=n/dur*60, n_uniq=len(set(toks)), ttr=len(set(toks))/max(n,1),
        fill=sum(t in FILL for t in toks), fill_rate=sum(t in FILL for t in toks)/max(n,1),
        rep=sum(toks[i]==toks[i-1] for i in range(1,n)), rep_rate=sum(toks[i]==toks[i-1] for i in range(1,n))/max(n,1),
        n_sent=len(sents), avg_sent_len=n/max(len(sents),1),
        pause_n=int((gaps>0.3).sum()), pause_long=int((gaps>1.0).sum()),
        pause_mean=float(gaps[gaps>0.3].mean()) if (gaps>0.3).any() else 0., pause_total=float(gaps[gaps>0.3].sum()),
        prob_mean=pr.mean(), prob_min=pr.min(), prob_low=(pr<0.5).mean(),
        avg_wlen=np.mean([len(t) for t in toks]) if n else 0,
        lang_p=j["lang_p"], is_en=int(j["lang"]=="en"), nll=nll, ppl=pp, speech_dur=dur,
    )

Ftr = pd.DataFrame([feats("train", f) for f in tqdm(train.filename)])
Fte = pd.DataFrame([feats("test",  f) for f in tqdm(test.filename)])
Ftr.to_parquet("/kaggle/working/hand_train.parquet"); Fte.to_parquet("/kaggle/working/hand_test.parquet")
print(Ftr.assign(label=train.label).corr()["label"].sort_values())

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

  0%|          | 0/769 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


  0%|          | 0/216 [00:00<?, ?it/s]

prob_low       -0.553722
nll            -0.213482
ppl            -0.205833
pause_mean     -0.172068
rep            -0.151149
rep_rate       -0.149062
fill_rate      -0.103486
avg_sent_len   -0.092687
pause_total    -0.084056
pause_long      0.013700
fill            0.029524
ttr             0.121446
is_en           0.126414
pause_n         0.137370
speech_dur      0.158622
prob_min        0.208217
n_sent          0.221628
avg_wlen        0.243259
wpm             0.271183
n_words         0.324571
lang_p          0.446750
n_uniq          0.461419
prob_mean       0.614133
label           1.000000
Name: label, dtype: float64


**CELL E: baseline cross-validation**


baseline cv plus first submission


In [10]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from scipy.stats import pearsonr
import lightgbm as lgb

y = train.label.values
strat = np.round(y).astype(int); strat[strat<=1] = 1       # merge rare classes
skf = StratifiedKFold(5, shuffle=True, random_state=42)
def rep(name, p): print(f"{name}: RMSE {np.sqrt(np.mean((p-y)**2)):.4f}  r {pearsonr(p,y)[0]:.4f}")

Atr = np.load("/kaggle/working/wavlm_train.npy"); Ate = np.load("/kaggle/working/wavlm_test.npy")
LAYERS = [6, 9, 12]                                         # tune later
Xa_tr = Atr[:, LAYERS].reshape(len(Atr), -1); Xa_te = Ate[:, LAYERS].reshape(len(Ate), -1)
Xh_tr, Xh_te = Ftr.values, Fte.values

def cv(make, Xtr, Xte):
    oof = np.zeros(len(y)); te = np.zeros(len(Xte))
    for tr, va in skf.split(Xtr, strat):
        m = make().fit(Xtr[tr], y[tr])
        oof[va] = m.predict(Xtr[va]); te += m.predict(Xte) / 5
    return np.clip(oof, 0, 5), np.clip(te, 0, 5)

o1, t1 = cv(lambda: make_pipeline(StandardScaler(), Ridge(alpha=3000)), Xa_tr, Xa_te); rep("audio ridge", o1)
o2, t2 = cv(lambda: lgb.LGBMRegressor(n_estimators=400, learning_rate=0.03, num_leaves=15,
                    subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1), Xh_tr, Xh_te); rep("hand lgbm", o2)
oe = (o1 + o2) / 2; rep("blend", oe); te = (t1 + t2) / 2

# submission: follow sample_submission rows
pred = dict(zip(test.filename, te))
out = sub[["filename"]].copy(); out["label"] = out.filename.map(pred)
print("missing:", out.label.isna().sum())
out.to_csv("/kaggle/working/submission.csv", index=False)

audio ridge: RMSE 0.5690  r 0.8899


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/v

hand lgbm: RMSE 0.7808  r 0.7776
blend: RMSE 0.6023  r 0.8786
missing: 179


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


**CELL F**
diagnose

In [11]:
# --- submission vs test vs train ---
sub_set, test_set, train_set = set(sub.filename), set(test.filename), set(train.filename)
print("sub in test.csv  :", len(sub_set & test_set))
print("sub in train.csv :", len(sub_set & train_set))
print("sub in test dir  :", sum(os.path.exists(f"{D}/test/{f}")  for f in sub.filename))
print("sub in train dir :", sum(os.path.exists(f"{D}/train/{f}") for f in sub.filename))
print("test.csv in train.csv:", len(test_set & train_set))
print("neither (sub only):", len(sub_set - test_set - train_set))

# same filename in train and test dirs: same audio or different?
common = sorted(set(os.listdir(f"{D}/train")) & set(os.listdir(f"{D}/test")))
print("same name in both dirs:", len(common))
same = 0
for f in common[:20]:
    a, _ = sf.read(f"{D}/train/{f}"); b, _ = sf.read(f"{D}/test/{f}")
    same += (len(a) == len(b)) and np.allclose(a, b)
print("identical audio among first 20:", same, "/ 20")

# --- the label-0 group ---
num = lambda s: int(s.split("_")[1].split(".")[0])
train["id"] = train.filename.map(num); test["id"] = test.filename.map(num)
print("train id range:", train.id.min(), train.id.max(), "| test:", test.id.min(), test.id.max())
big = train.id >= 5000
print("train id>=5000:", big.sum()); print(train[big].label.value_counts().sort_index())
print("label 0 with id<5000:", ((train.label == 0) & ~big).sum())
print("test id>=5000:", (test.id >= 5000).sum())

# language / transcript length of label-0 vs rest
rows = []
for f, l in zip(train.filename, train.label):
    j = json.load(open(f"{OUT}/train_{f}.json"))
    rows.append((l, j["lang"], j["lang_p"], len(j["words"])))
t = pd.DataFrame(rows, columns=["label", "lang", "lang_p", "nwords"])
print(t.groupby(t.label == 0)[["lang_p", "nwords"]].mean())
print(t[t.label == 0].lang.value_counts())

sub in test.csv  : 25
sub in train.csv : 105
sub in test dir  : 25
sub in train dir : 105
test.csv in train.csv: 212
neither (sub only): 99
same name in both dirs: 212
identical audio among first 20: 0 / 20
train id range: 0 5073 | test: 0 215
train id>=5000: 37
label
0.0    37
Name: count, dtype: int64
label 0 with id<5000: 0
test id>=5000: 0
         lang_p      nwords
label                      
False  0.875773  101.479508
True   0.611797   86.864865
lang
en    30
nn     4
tl     1
ms     1
hi     1
Name: count, dtype: int64


**CELL G**
diagnose plus improve


In [12]:
fn = lambda s: int(s.split("_")[1].split(".")[0])
big = (train.filename.map(fn) >= 5000).values
nz = y > 0
rm = lambda p, m: np.sqrt(np.mean((p[m] - y[m])**2))
for n, p in [("audio", o1), ("hand", o2), ("blend", oe)]:
    print(f"{n}: all {rm(p, np.ones(len(y), bool)):.3f} | label>0 {rm(p, nz):.3f} | label0 {rm(p, ~nz):.3f}")
print("id>=5000 count:", big.sum(), "| of those label0:", (big & ~nz).sum())
print(pd.DataFrame({"y": y, "abs_err": np.abs(o1 - y)}).groupby("y").abs_err.agg(["mean", "count"]))

# layer + alpha sweep (audio ridge)
for L in [[12], [6, 9, 12], [3, 6, 9, 12], list(range(13))]:
    Xa = Atr[:, L].reshape(len(Atr), -1)
    for a in [1000, 3000, 10000, 30000]:
        o, _ = cv(lambda: make_pipeline(StandardScaler(), Ridge(alpha=a)), Xa, Ate[:, L].reshape(len(Ate), -1))
        print(L, a, f"RMSE {rm(o, np.ones(len(y), bool)):.4f}  >0: {rm(o, nz):.4f}")

# stack: nonneg weights on OOF
from scipy.optimize import nnls
w, _ = nnls(np.c_[o1, o2], y); print("weights audio/hand:", w)
os_ = np.c_[o1, o2] @ w; print("stack RMSE", rm(np.clip(os_, 0, 5), np.ones(len(y), bool)))

audio: all 0.569 | label>0 0.582 | label0 0.153
hand: all 0.781 | label>0 0.729 | label0 1.467
blend: all 0.602 | label>0 0.592 | label0 0.772
id>=5000 count: 37 | of those label0: 37
         mean  count
y                   
0.0  0.087333     37
1.0  1.959821      1
1.5  0.911139      3
2.0  0.744897    102
2.5  0.379425     79
3.0  0.329713    174
3.5  0.402354     64
4.0  0.405445    124
4.5  0.502834     52
5.0  0.455925    133
[12] 1000 RMSE 0.6094  >0: 0.6234
[12] 3000 RMSE 0.6618  >0: 0.6767
[12] 10000 RMSE 0.7500  >0: 0.7666
[12] 30000 RMSE 0.8328  >0: 0.8504
[6, 9, 12] 1000 RMSE 0.5420  >0: 0.5544
[6, 9, 12] 3000 RMSE 0.5690  >0: 0.5821
[6, 9, 12] 10000 RMSE 0.6222  >0: 0.6366
[6, 9, 12] 30000 RMSE 0.6987  >0: 0.7146
[3, 6, 9, 12] 1000 RMSE 0.5386  >0: 0.5510
[3, 6, 9, 12] 3000 RMSE 0.5614  >0: 0.5746
[3, 6, 9, 12] 10000 RMSE 0.6115  >0: 0.6257
[3, 6, 9, 12] 30000 RMSE 0.6856  >0: 0.7014
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] 1000 RMSE 0.5297  >0: 0.5424
[0, 1, 2, 3, 4, 5,

**CELL H**

In [13]:
Aall_tr = Atr.reshape(len(Atr), -1); Aall_te = Ate.reshape(len(Ate), -1)   # all 13 layers
allm = np.ones(len(y), bool)

res = {}
for a in [100, 200, 300, 500, 1000, 2000]:
    o, t = cv(lambda: make_pipeline(StandardScaler(), Ridge(alpha=a)), Aall_tr, Aall_te)
    res[a] = (o, t); print("ridge", a, f"RMSE {rm(o, allm):.4f}  >0 {rm(o, nz):.4f}")
best_a = min(res, key=lambda a: rm(res[a][0], allm)); print("best alpha", best_a)
o1b, t1b = res[best_a]

# second audio model: PCA + SVR (different inductive bias)
from sklearn.decomposition import PCA
from sklearn.svm import SVR
print("for normal SVR")
for C in [1, 3, 10]:
    o, t = cv(lambda: make_pipeline(StandardScaler(), PCA(256, random_state=0), SVR(C=C, epsilon=0.1)), Aall_tr, Aall_te)
    res[f"svr{C}"] = (o, t); print("svr", C, f"RMSE {rm(o, allm):.4f}")
    


# SVR edge check
print("for SVR Edge")
for C in [30, 100]:
    o, t = cv(lambda: make_pipeline(StandardScaler(), PCA(256, random_state=0), SVR(C=C, epsilon=0.1)), Aall_tr, Aall_te)
    res[f"svr{C}"] = (o, t); print("svr", C, f"RMSE {rm(o, allm):.4f}")


best_s = min([k for k in res if str(k).startswith("svr")], key=lambda k: rm(res[k][0], allm))
o3, t3 = res[best_s]

# stack: audio ridge + audio svr + hand lgbm
O = np.c_[o1b, o3, o2]; T = np.c_[t1b, t3, t2]
w, _ = nnls(O, y); print("weights ridge/svr/hand:", w)
os_ = np.clip(O @ w, 0, 5); print("stack RMSE", rm(os_, allm), "| >0", rm(os_, nz))

ridge 100 RMSE 0.5428  >0 0.5557
ridge 200 RMSE 0.5391  >0 0.5519
ridge 300 RMSE 0.5365  >0 0.5492
ridge 500 RMSE 0.5330  >0 0.5457
ridge 1000 RMSE 0.5297  >0 0.5424
ridge 2000 RMSE 0.5308  >0 0.5436
best alpha 1000
for normal SVR
svr 1 RMSE 0.5686
svr 3 RMSE 0.5353
svr 10 RMSE 0.5302
for SVR Edge
svr 30 RMSE 0.5331
svr 100 RMSE 0.5331
weights ridge/svr/hand: [0.42907117 0.4299003  0.14136366]
stack RMSE 0.511027481590083 | >0 0.5202097085676654


In [14]:
np.save("/kaggle/working/oof_audio.npy", O); np.save("/kaggle/working/te_audio.npy", T)

In [15]:
assert len(test) == 216 and len(Ate) == 216 and len(Fte) == 216
chk = [0, 100, 215]
for i in chk:
    e = emb(f"{D}/test/{test.filename[i]}")            # recompute WavLM for that file
    print(i, test.filename[i], "wavlm match:", np.allclose(e, Ate[i], atol=1e-2))
    h = feats("test", test.filename[i])
    print("   hand match:", np.allclose(pd.Series(h).values.astype(float), Fte.iloc[i].values.astype(float), equal_nan=True, atol=1e-3))

0 audio_128.wav wavlm match: True
   hand match: True
100 audio_7.wav wavlm match: True
   hand match: True
215 audio_170.wav wavlm match: True
   hand match: True


**CELL I**

In [16]:
import torch, torch.nn as nn, random
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
import gc;gc.collect(); torch.cuda.empty_cache()
def load_text(split, files):
    return [json.load(open(f"{OUT}/{split}_{f}.json"))["text"] for f in files]

train = pd.read_csv(f"{D}/train.csv"); test = pd.read_csv(f"{D}/test.csv")
assert len(train) == 769 and len(test) == 216 and "audio_436.wav" not in set(test.filename)
y = train.label.values
strat = np.round(y).astype(int); strat[strat <= 1] = 1
Ttr, Tte = load_text("train", train.filename), load_text("test", test.filename)
print("Sample:", Ttr[0][:200])

MN, ML, BS, EP, LR = "microsoft/deberta-v3-base", 320, 16, 4, 2e-5
tk = AutoTokenizer.from_pretrained(MN)
dev0 = "cuda:0"

class Reg(nn.Module):
    def __init__(s):
        super().__init__()
        s.b = AutoModel.from_pretrained(MN, dtype=torch.float32)
        s.h = nn.Linear(s.b.config.hidden_size, 1)
    def forward(s, ids, am):
        x = s.b(input_ids=ids, attention_mask=am).last_hidden_state
        m = am.unsqueeze(-1).float()
        x = (x * m).sum(1) / m.sum(1)
        return s.h(x).squeeze(-1)

def enc(texts):
    e = tk(texts, truncation=True, max_length=ML, padding="max_length", return_tensors="pt")
    return e.input_ids, e.attention_mask

Itr, Mtr = enc(Ttr); Ite, Mte = enc(Tte)
Y = torch.tensor(y / 5.0, dtype=torch.float32)

def predict(m, I, M):
    m.eval(); out = []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for i in range(0, len(I), 32):
            out.append(m(I[i:i+32].to(dev0), M[i:i+32].to(dev0)).float().cpu())
    return torch.cat(out).numpy() * 5

oof_d = np.zeros(len(y)); te_d = np.zeros(len(Tte))
for k, (tr, va) in enumerate(skf.split(np.zeros(len(y)), strat)):
    torch.manual_seed(k); random.seed(k); np.random.seed(k)
    m = Reg().to(dev0).float()
    opt = torch.optim.AdamW(
        [{"params": m.b.parameters(), "lr": LR}, {"params": m.h.parameters(), "lr": 1e-3}],
        weight_decay=0.01)
    steps = EP * (len(tr) // BS)
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    sc = torch.amp.GradScaler("cuda")
    best, bp, bt = 9, None, None
    for ep in range(EP):
        m.train(); perm = np.random.permutation(tr)
        for i in range(0, len(perm) - BS + 1, BS):
            b = perm[i:i+BS]
            with torch.autocast("cuda", dtype=torch.float16):
                p = m(Itr[b].to(dev0), Mtr[b].to(dev0))
            loss = nn.functional.mse_loss(p.float(), Y[b].to(dev0))
            opt.zero_grad()
            sc.scale(loss).backward()
            sc.unscale_(opt)
            nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            sc.step(opt); sc.update(); sch.step()
        pv = predict(m, Itr[va], Mtr[va])
        r = np.sqrt(np.mean((np.clip(pv, 0, 5) - y[va])**2))
        print(f"fold {k} ep {ep} RMSE {r:.4f}")
        if r < best: best, bp, bt = r, pv, predict(m, Ite, Mte)
    oof_d[va] = bp; te_d += bt / 5
    del m, opt; torch.cuda.empty_cache()

oof_d, te_d = np.clip(oof_d, 0, 5), np.clip(te_d, 0, 5)
np.save("/kaggle/working/oof_deb.npy", oof_d); np.save("/kaggle/working/te_deb.npy", te_d)
print("DeBERTa OOF RMSE", rm(oof_d, allm), "| >0", rm(oof_d, nz))

Sample: Um, well, this, the playground is very, very beautiful and fun.  There are, there are places like the merry-go-round, the swing, the students usually enjoy the  swing.  it has to be like, it's more li


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

...

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


fold 3 ep 0 RMSE 1.0358
fold 3 ep 1 RMSE 1.0649
fold 3 ep 2 RMSE 1.2067
fold 3 ep 3 RMSE 0.9238


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


fold 4 ep 0 RMSE 0.9606
fold 4 ep 1 RMSE 1.2233
fold 4 ep 2 RMSE 0.9306
fold 4 ep 3 RMSE 0.9145
DeBERTa OOF RMSE 0.8735706942203565 | >0 0.7330650038133324


In [17]:
np.save("/kaggle/working/oof_audio.npy", O); np.save("/kaggle/working/te_audio.npy", T)


In [18]:
O2 = np.c_[O, oof_d]; T2 = np.c_[T, te_d]
w2, _ = nnls(O2, y); print("weights ridge/svr/hand/deb:", w2)
os2 = np.clip(O2 @ w2, 0, 5)
print("stack RMSE", rm(os2, allm), "| >0", rm(os2, nz))   # compare 0.5104 / 0.5192

weights ridge/svr/hand/deb: [0.38801125 0.42392448 0.04021394 0.15146852]
stack RMSE 0.49955716870705685 | >0 0.5018862795741476


In [19]:
te_final = np.clip(T2 @ w2, 0, 5)
out216 = pd.DataFrame({"filename": test.filename, "label": te_final})
out216.to_csv("/kaggle/working/submission_test216.csv", index=False)
print(out216.shape, out216.label.describe())

(216, 2) count    216.000000
mean       3.293651
std        0.782016
min        1.716802
25%        2.742094
50%        3.225114
75%        3.738991
max        5.000000
Name: label, dtype: float64


In [20]:
sub_set, test_set, train_set = set(sub.filename), set(test.filename), set(train.filename)
print("sub in test.csv:", len(sub_set & test_set), "| in train.csv:", len(sub_set & train_set))
print("sub in test dir:", sum(os.path.exists(f"{D}/test/{f}") for f in sub.filename))
print("sub in train dir:", sum(os.path.exists(f"{D}/train/{f}") for f in sub.filename))
num = lambda s: int(s.split("_")[1].split(".")[0])
print("test id>=5000:", sum(num(f) >= 5000 for f in test.filename))
print(sub.filename.head(10).tolist())

sub in test.csv: 25 | in train.csv: 105
sub in test dir: 25
sub in train dir: 105
test id>=5000: 0
['audio_804.wav', 'audio_1028.wav', 'audio_865.wav', 'audio_774.wav', 'audio_1138.wav', 'audio_278.wav', 'audio_1212.wav', 'audio_178.wav', 'audio_542.wav', 'audio_248.wav']


In [21]:
te_final = np.clip(T2 @ w2, 0, 5)
final = pd.DataFrame({"filename": test.filename, "label": te_final})
assert len(final) == 216 and final.label.notna().all() and final.filename.is_unique
final.to_csv("/kaggle/working/submission.csv", index=False)
print(final.label.describe())

count    216.000000
mean       3.293651
std        0.782016
min        1.716802
25%        2.742094
50%        3.225114
75%        3.738991
max        5.000000
Name: label, dtype: float64


In [22]:
def cv_drop(make, Xtr, Xte, drop):
    oof = np.zeros(len(y)); te = np.zeros(len(Xte))
    for tr, va in skf.split(Xtr, strat):
        tr2 = tr[~drop[tr]]
        m = make().fit(Xtr[tr2], y[tr2])
        oof[va] = m.predict(Xtr[va]); te += m.predict(Xte) / 5
    return np.clip(oof, 0, 5), np.clip(te, 0, 5)

big = (train.filename.map(lambda s: int(s.split("_")[1].split(".")[0])) >= 5000).values
oa, ta = cv_drop(lambda: make_pipeline(StandardScaler(), Ridge(alpha=1000)), Aall_tr, Aall_te, big)
print("ridge drop50xx  >0:", rm(oa, nz), "| keep >0: 0.5424")

ridge drop50xx  >0: 0.543247586631864 | keep >0: 0.5424


**Re-Running DeBERTa with Epochs=8 and learning rate = 3e-5**

In [23]:
import torch, torch.nn as nn, random
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
import gc;gc.collect(); torch.cuda.empty_cache()
def load_text(split, files):
    return [json.load(open(f"{OUT}/{split}_{f}.json"))["text"] for f in files]

train = pd.read_csv(f"{D}/train.csv"); test = pd.read_csv(f"{D}/test.csv")
assert len(train) == 769 and len(test) == 216 and "audio_436.wav" not in set(test.filename)
y = train.label.values
strat = np.round(y).astype(int); strat[strat <= 1] = 1
Ttr, Tte = load_text("train", train.filename), load_text("test", test.filename)
print("Sample:", Ttr[0][:200])

MN, ML, BS, EP, LR = "microsoft/deberta-v3-base", 320, 16, 8, 3e-5
tk = AutoTokenizer.from_pretrained(MN)
dev0 = "cuda:0"

class Reg(nn.Module):
    def __init__(s):
        super().__init__()
        s.b = AutoModel.from_pretrained(MN, dtype=torch.float32)
        s.h = nn.Linear(s.b.config.hidden_size, 1)
    def forward(s, ids, am):
        x = s.b(input_ids=ids, attention_mask=am).last_hidden_state
        m = am.unsqueeze(-1).float()
        x = (x * m).sum(1) / m.sum(1)
        return s.h(x).squeeze(-1)

def enc(texts):
    e = tk(texts, truncation=True, max_length=ML, padding="max_length", return_tensors="pt")
    return e.input_ids, e.attention_mask

Itr, Mtr = enc(Ttr); Ite, Mte = enc(Tte)
Y = torch.tensor(y / 5.0, dtype=torch.float32)

def predict(m, I, M):
    m.eval(); out = []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for i in range(0, len(I), 32):
            out.append(m(I[i:i+32].to(dev0), M[i:i+32].to(dev0)).float().cpu())
    return torch.cat(out).numpy() * 5

oof_d = np.zeros(len(y)); te_d = np.zeros(len(Tte))
for k, (tr, va) in enumerate(skf.split(np.zeros(len(y)), strat)):
    torch.manual_seed(k); random.seed(k); np.random.seed(k)
    m = Reg().to(dev0).float()
    opt = torch.optim.AdamW(
        [{"params": m.b.parameters(), "lr": LR}, {"params": m.h.parameters(), "lr": 1e-3}],
        weight_decay=0.01)
    steps = EP * (len(tr) // BS)
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    sc = torch.amp.GradScaler("cuda")
    best, bp, bt = 9, None, None
    for ep in range(EP):
        m.train(); perm = np.random.permutation(tr)
        for i in range(0, len(perm) - BS + 1, BS):
            b = perm[i:i+BS]
            with torch.autocast("cuda", dtype=torch.float16):
                p = m(Itr[b].to(dev0), Mtr[b].to(dev0))
            loss = nn.functional.mse_loss(p.float(), Y[b].to(dev0))
            opt.zero_grad()
            sc.scale(loss).backward()
            sc.unscale_(opt)
            nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            sc.step(opt); sc.update(); sch.step()
        pv = predict(m, Itr[va], Mtr[va])
        r = np.sqrt(np.mean((np.clip(pv, 0, 5) - y[va])**2))
        print(f"fold {k} ep {ep} RMSE {r:.4f}")
        if r < best: best, bp, bt = r, pv, predict(m, Ite, Mte)
    oof_d[va] = bp; te_d += bt / 5
    del m, opt; torch.cuda.empty_cache()

oof_d, te_d = np.clip(oof_d, 0, 5), np.clip(te_d, 0, 5)
np.save("/kaggle/working/oof_deb8.npy", oof_d); np.save("/kaggle/working/te_deb8.npy", te_d)
print("DeBERTa OOF RMSE", rm(oof_d, allm), "| >0", rm(oof_d, nz))

Sample: Um, well, this, the playground is very, very beautiful and fun.  There are, there are places like the merry-go-round, the swing, the students usually enjoy the  swing.  it has to be like, it's more li


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_23/3827754262.py:65: Us

fold 0 ep 0 RMSE 1.3941
fold 0 ep 1 RMSE 0.8885
fold 0 ep 2 RMSE 1.3546
fold 0 ep 3 RMSE 0.9472
fold 0 ep 4 RMSE 0.8504
fold 0 ep 5 RMSE 1.0502
fold 0 ep 6 RMSE 0.8963
fold 0 ep 7 RMSE 0.8751


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

...

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


fold 3 ep 0 RMSE 0.9794
fold 3 ep 1 RMSE 0.9654
fold 3 ep 2 RMSE 0.9247
fold 3 ep 3 RMSE 0.9195
fold 3 ep 4 RMSE 1.0724
fold 3 ep 5 RMSE 0.9097
fold 3 ep 6 RMSE 0.9168
fold 3 ep 7 RMSE 0.9119


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


fold 4 ep 0 RMSE 1.0805
fold 4 ep 1 RMSE 1.0342
fold 4 ep 2 RMSE 1.0276
fold 4 ep 3 RMSE 1.0889
fold 4 ep 4 RMSE 0.9720
fold 4 ep 5 RMSE 0.8611
fold 4 ep 6 RMSE 0.8869
fold 4 ep 7 RMSE 0.8825
DeBERTa OOF RMSE 0.8688676666335352 | >0 0.7280235646519679


In [24]:
oof_d8, te_d8 = np.load("/kaggle/working/oof_deb8.npy"), np.load("/kaggle/working/te_deb8.npy")
O3 = np.c_[O, oof_d, oof_d8]; T3 = np.c_[T, te_d, te_d8]
w3, _ = nnls(O3, y); print("weights ridge/svr/hand/deb4/deb8:", w3)
os3 = np.clip(O3 @ w3, 0, 5); print("stack", rm(os3, allm), "| >0", rm(os3, nz))   # beat 0.5016 / 0.5048

weights ridge/svr/hand/deb4/deb8: [0.38249828 0.42802369 0.04620796 0.14613513 0.        ]
stack 0.50038860928581 | >0 0.5029158068403332


In [25]:
a = np.load("/kaggle/working/oof_deb.npy"); b = np.load("/kaggle/working/oof_deb8.npy")
print("deb.npy == deb8.npy:", np.allclose(a, b), "| deb.npy == in-memory oof_d:", np.allclose(a, oof_d))

deb.npy == deb8.npy: False | deb.npy == in-memory oof_d: False


In [26]:
d4 = np.load("/kaggle/working/oof_deb.npy");  t4 = np.load("/kaggle/working/te_deb.npy")
d8 = np.load("/kaggle/working/oof_deb8.npy"); t8 = np.load("/kaggle/working/te_deb8.npy")
print("deb4 OOF", rm(d4, allm), "| deb8 OOF", rm(d8, allm), "| corr", np.corrcoef(d4, d8)[0, 1])

O3 = np.c_[O, d4, d8]; T3 = np.c_[T, t4, t8]
w3, _ = nnls(O3, y); print("weights ridge/svr/hand/deb4/deb8:", w3)
os3 = np.clip(O3 @ w3, 0, 5); print("stack", rm(os3, allm), "| >0", rm(os3, nz))

deb4 OOF 0.8735706942203565 | deb8 OOF 0.8688676666335352 | corr 0.964952800795143
weights ridge/svr/hand/deb4/deb8: [0.38535135 0.42484871 0.03889839 0.11531665 0.03909635]
stack 0.49944890605593634 | >0 0.5015735149353213


In [27]:
te_final = np.clip(T3 @ w3, 0, 5)
final = pd.DataFrame({"filename": test.filename, "label": te_final})
assert final.shape == (216, 2) and final.label.notna().all() and set(final.filename) == set(test.filename)
final.to_csv("/kaggle/working/submission.csv", index=False)
print(final.label.describe())

count    216.000000
mean       3.293346
std        0.782229
min        1.710199
25%        2.748268
50%        3.221764
75%        3.740527
max        5.000000
Name: label, dtype: float64


In [28]:
f = pd.read_csv("/kaggle/working/submission.csv")
print(f.shape, f.columns.tolist(), f.label.min(), f.label.max())   # (216, 2) ['filename','label'] in 0-5

(216, 2) ['filename', 'label'] 1.7101991759060824 5.0
